In [1]:
import torch
from sentence_transformers import SentenceTransformer, InputExample, losses
from sentence_transformers.evaluation import EmbeddingSimilarityEvaluator, SimilarityFunction
from torch.utils.data import DataLoader
from datasets import load_dataset
import pandas as pd
from sklearn.model_selection import train_test_split

/raid/deallab/anaconda3/envs/djk/lib/python3.11/site-packages/sentence_transformers/cross_encoder/CrossEncoder.py:13: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


In [ ]:
model_path = "fine_tuned_model"

# 모델 로드
model = SentenceTransformer(model_path)

# 모델을 GPU로 이동 (가능한 경우)
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

In [3]:
# parquet 파일 로드
file_path = '/raid/deallab/SF_RAG_Data/ASQA/dev.parquet'
df = pd.read_parquet(file_path)

# 데이터 확인
df.head()

,ambiguous_question,qa_pairs,wikipages,annotations,sample_id
0,Who has the highest goals in world football?,"[{'context': 'No context provided', 'question'...",[{'title': 'International Federation of Footba...,"[{'knowledge': [], 'long_answer': 'Ali Dael ha...",-7013890438520559398
1,Who is the original artist of sound of silence?,[{'context': 'Sounds of Silence is the second ...,"[{'title': 'The Sound of Silence', 'url': 'htt...",[{'knowledge': [{'content': 'Wednesday Morning...,7089015503030534342
2,When was the first apple i phone made?,"[{'context': 'On January 9, 2007, Steve Jobs a...","[{'title': 'iPhone (1st generation)', 'url': '...",[{'knowledge': [{'content': 'The iPhone was re...,8793099883447006698
3,Who played the weasley brothers in harry potter?,[{'context': 'Richard Fish appeared as Bill br...,"[{'title': 'List of Harry Potter characters', ...",[{'knowledge': [{'content': 'Dozens of actors ...,-881464876144297194
4,How many state parks are there in virginia?,"[{'context': 'No context provided', 'question'...","[{'title': 'List of Virginia state parks', 'ur...",[{'knowledge': [{'content': 'Virginia opened i...,1650309494326541834


In [13]:
dif_questions=[]
for i in range(len(df)):
    res=[]
    for data in df['qa_pairs'][i]:
        res.append(data['question'])
    dif_questions.append(res)

In [14]:
dif_questions

[["Who has the highest goals in men's world international football?",
  "Who has the highest goals all-time in men's football?",
  "Who has the highest goals in women's world international football?"],
 ['Who is the original artist of sound of silence, the song, released in 1964?',
  'Who is the original artist of sound of silence, the album?',
  'Who is the original artist of sound of silence, the song, released in 2016?'],
 ['When was the first apple i phone released?',
  'When was the first apple i phone for beta testing made?',
  'When was the first apple i phone 1 made?',
  'When was the first apple i phone beta made?'],
 ['Who played  Bill weasley in Harry Potter and the Prisoner of Azkaban?',
  'Who played percy weasley in harry potter?',
  'Who played fred weasley in harry potter?',
  'Who played ron weasley in harry potter?',
  'Who played george weasley in harry potter?',
  'Who played  Bill weasley in harry potter (2001-2011)?'],
 ['How many state parks are there in virginia

In [15]:
df['dif_questions']=dif_questions

In [16]:
df.head()

,ambiguous_question,qa_pairs,wikipages,annotations,sample_id,dif_questions
0,Who has the highest goals in world football?,"[{'context': 'No context provided', 'question'...",[{'title': 'International Federation of Footba...,"[{'knowledge': [], 'long_answer': 'Ali Dael ha...",-7013890438520559398,[Who has the highest goals in men's world inte...
1,Who is the original artist of sound of silence?,[{'context': 'Sounds of Silence is the second ...,"[{'title': 'The Sound of Silence', 'url': 'htt...",[{'knowledge': [{'content': 'Wednesday Morning...,7089015503030534342,[Who is the original artist of sound of silenc...
2,When was the first apple i phone made?,"[{'context': 'On January 9, 2007, Steve Jobs a...","[{'title': 'iPhone (1st generation)', 'url': '...",[{'knowledge': [{'content': 'The iPhone was re...,8793099883447006698,"[When was the first apple i phone released?, W..."
3,Who played the weasley brothers in harry potter?,[{'context': 'Richard Fish appeared as Bill br...,"[{'title': 'List of Harry Potter characters', ...",[{'knowledge': [{'content': 'Dozens of actors ...,-881464876144297194,[Who played Bill weasley in Harry Potter and ...
4,How many state parks are there in virginia?,"[{'context': 'No context provided', 'question'...","[{'title': 'List of Virginia state parks', 'ur...",[{'knowledge': [{'content': 'Virginia opened i...,1650309494326541834,[How many state parks are there in virginia in...


In [18]:
dev_df=df[['ambiguous_question','dif_questions']]
dev_df

,ambiguous_question,dif_questions
0,Who has the highest goals in world football?,[Who has the highest goals in men's world inte...
1,Who is the original artist of sound of silence?,[Who is the original artist of sound of silenc...
2,When was the first apple i phone made?,"[When was the first apple i phone released?, W..."
3,Who played the weasley brothers in harry potter?,[Who played Bill weasley in Harry Potter and ...
4,How many state parks are there in virginia?,[How many state parks are there in virginia in...
...,...,...
943,Where did the practice of baptism come from?,"[Where did the baptism practice come from?, Wh..."
944,Who sings the song i'm just a love machine?,"[Who sings the song ""Love Machine"" from 1975?,..."
945,When was the last time man united were in the ...,"[As of the 2016-2017 season, when was the last..."
946,Who sings beautiful girl in singin in the rain?,[Who sings beautiful girl in the 1952 stage mu...


In [ ]:
query=dev_df['ambiguous_question']
dif_queries=dev_df['dif_questions']

In [ ]:
def search_documents(query, documents, model, device="cuda"):
    # 질문 임베딩 생성 (GPU로 이동)
    query_embedding = model.encode(query, convert_to_tensor=True).to(device)
    # 문서 임베딩 생성 (GPU로 이동)
    document_embeddings = model.encode(documents, convert_to_tensor=True).to(device)
    
    # 질문과 문서 간 코사인 유사도 계산 (PyTorch 기반)
    query_embedding = query_embedding.unsqueeze(0)  # 배치 차원 추가
    similarities = torch.nn.functional.cosine_similarity(query_embedding, document_embeddings)

    # 유사도에 따라 문서 정렬
    top_results = similarities.argsort(descending=True)[:5]
    print(top_results)
    res={}
    for idx in top_results:
        tmp=documents[idx]
        res[tmp]=similarities[idx]
    return list(res.items())

# 예시 사용법
for i in range(20):
    print(f"Query {i+1} : {question[i]}")
    print("-"*100)
    results = search_documents(question[i], context, model)
    print("Retrieved doc :")
    for j in range(len(results)):
        print(f"\tRank {j} : {results[j]}")
    print("Original doc :", context2[i], "\n")
